# Task 4 - CNN Pipeline
Converted from Task_3/CNN.py to a Jupyter notebook.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.errors import NotGeoreferencedWarning

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms

In [2]:
warnings.filterwarnings("ignore", category=NotGeoreferencedWarning)

# Paths
TASK_DIR = Path.cwd()
BASE_DIR = TASK_DIR.parent
SAMPLES_DIR = BASE_DIR / "samples"
LABELS_DIR = BASE_DIR / "labels"
OUTPUT_DIR = TASK_DIR / "outputs"

# Simple settings
IMAGE_SIZE = 64 
BATCH_SIZE = 16 
EPOCHS = 30 
LEARNING_RATE = 0.001 
PATIENCE = 10
SEED = 42

# NDVI thresholds for 3 classes
LOW_THRESHOLD = 150
MEDIUM_THRESHOLD = 180
CLASS_NAMES = ["low", "medium", "high"]

np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
def label_to_class(label_tile):
    mean_ndvi = float(np.nanmean(label_tile))

    if mean_ndvi < LOW_THRESHOLD:
        return 0
    if mean_ndvi < MEDIUM_THRESHOLD:
        return 1
    return 2


# -----------------------------------------------------------------------------
# Load the TIFF data
# -----------------------------------------------------------------------------

images = []
labels = []

for sample_path in sorted(SAMPLES_DIR.glob("*.tif*")):
    label_path = LABELS_DIR / sample_path.name.replace("img_", "ndvi_")

    if not label_path.exists():
        continue

    with rasterio.open(sample_path) as src:
        image = src.read(
            out_shape=(src.count, IMAGE_SIZE, IMAGE_SIZE),
            resampling=Resampling.bilinear,
        ).astype(np.float32)

    with rasterio.open(label_path) as src:
        label = src.read(1).astype(np.float32)

    image = image / 255.0
    image = np.nan_to_num(image)

    images.append(image)
    labels.append(label_to_class(label))

images = np.array(images, dtype=np.float32)
labels = np.array(labels, dtype=np.int64)

if len(images) == 0:
    raise RuntimeError("No image/label pairs were loaded.")

print(f"Total images: {len(images)}")
print(f"Class counts: {dict(zip(CLASS_NAMES, np.bincount(labels, minlength=3)))}")

Total images: 614
Class counts: {'low': np.int64(27), 'medium': np.int64(338), 'high': np.int64(249)}


In [4]:
# -----------------------------------------------------------------------------
# Split into train / validation / test
# -----------------------------------------------------------------------------

train_images, temp_images, train_labels, temp_labels = train_test_split(
    images,
    labels,
    test_size=0.30,
    stratify=labels,
    random_state=SEED,
)

val_images, test_images, val_labels, test_labels = train_test_split(
    temp_images,
    temp_labels,
    test_size=0.50,
    stratify=temp_labels,
    random_state=SEED,
)

print(f"Train: {len(train_images)}")
print(f"Validation: {len(val_images)}")
print(f"Test: {len(test_images)}")

# Normalize using only the training set
mean = train_images.mean(axis=(0, 2, 3), keepdims=True)
std = train_images.std(axis=(0, 2, 3), keepdims=True)
std[std == 0] = 1.0

train_images = (train_images - mean) / std
val_images = (val_images - mean) / std
test_images = (test_images - mean) / std

Train: 429
Validation: 92
Test: 93


In [5]:
# -----------------------------------------------------------------------------
# Data Augmentation (only for training) with pytorch 
# -----------------------------------------------------------------------------

train_transforms = transforms.Compose([
    # transforms.RandomRotation(5),
    transforms.RandomHorizontalFlip(p=0.5),
    # transforms.RandomVerticalFlip(),
    # transforms.ColorJitter(brightness=0.05, contrast=0.05),
    # transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
])

In [6]:
# Custom Dataset class to apply transforms
class AugmentedDataset(torch.utils.data.Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = torch.tensor(self.images[idx], dtype=torch.float32)
        label = torch.tensor(self.labels[idx], dtype=torch.long)

        if self.transform:
            image = self.transform(image)

        return image, label


# Convert to PyTorch datasets with augmentation
train_data = AugmentedDataset(train_images, train_labels, transform=train_transforms)
val_data = AugmentedDataset(val_images, val_labels, transform=None)
test_data = AugmentedDataset(test_images, test_labels, transform=None)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)


# -----------------------------------------------------------------------------
# CNN model
# -----------------------------------------------------------------------------

class VegetationCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)   # 16 filters, 3x3 kernel, padding to keep size
        self.bn1 = nn.BatchNorm2d(16)                             # Batch normalization is added after convolution to stabilize training and improve convergence
        self.pool = nn.MaxPool2d(2, 2)                            # Max pooling reduces spatial dimensions by half, helping to capture features at different scales and reduce overfitting

        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)  # 32 filters, 3x3 kernel, padding to keep size
        self.bn2 = nn.BatchNorm2d(32)                             # Batch normalization is added after convolution to stabilize training and improve convergence

        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)  # 64 filters, 3x3 kernel, padding to keep size
        self.bn3 = nn.BatchNorm2d(64)                             # Batch normalization is added after convolution to stabilize training and improve convergence

        self.gap = nn.AdaptiveAvgPool2d(1)                        # Global Average Pooling reduces each feature map to a single value, helping to reduce overfitting and capture global features
        self.fc1 = nn.Linear(64, 32)                              # Fully connected layer to learn complex relationships between features
        self.dropout = nn.Dropout(p=0.2)                          # Dropout is added to prevent overfitting by randomly setting a fraction of the input units to 0 during training
        self.fc2 = nn.Linear(32, 3)                               # Output layer with 3 classes (low, medium, high)

    def forward(self, x):                                         # The forward method defines the data flow through the network. It applies convolutional layers with ReLU activation and batch normalization, followed by pooling, global average pooling, and fully connected layers with dropout for regularization.
        x = self.pool(F.relu(self.bn1(self.conv1(x))))            # First convolutional layer followed by batch normalization, ReLU activation, and max pooling
        x = self.pool(F.relu(self.bn2(self.conv2(x))))            # Second convolutional layer followed by batch normalization, ReLU activation, and max pooling
        x = F.relu(self.bn3(self.conv3(x)))                       # Third convolutional layer followed by batch normalization and ReLU activation (no pooling here to retain more spatial information)
        x = self.gap(x)                                           # Global Average Pooling to reduce each feature map to a single value
        x = torch.flatten(x, 1)                                   # Flatten the output from the convolutional layers to feed into the fully connected layers
        x = F.relu(self.fc1(x))                                   # First fully connected layer with ReLU activation
        x = self.dropout(x)                                       # Dropout for regularization to prevent overfitting
        x = self.fc2(x)                                           # Output layer that produces the final class scores for low, medium, and high vegetation density
        return x                                                  # The output of the forward method is the raw class scores (logits) for each of the three classes, which will be used with a loss function like CrossEntropyLoss during training.


net = VegetationCNN().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

In [7]:
# Verify augmentation is active on training samples across multiple trials
num_trials = 10
non_zero_orig_aug = 0
non_zero_aug_aug = 0

for i in range(num_trials):
    orig = torch.tensor(train_images[0], dtype=torch.float32)
    aug1, _ = train_data[0]
    aug2, _ = train_data[0]

    diff_orig_aug1 = (orig - aug1).abs().mean().item()
    diff_aug1_aug2 = (aug1 - aug2).abs().mean().item()

    if diff_orig_aug1 > 0:
        non_zero_orig_aug += 1
    if diff_aug1_aug2 > 0:
        non_zero_aug_aug += 1

    print(f"Trial {i+1:02d}: orig->aug1={diff_orig_aug1:.6f}, aug1->aug2={diff_aug1_aug2:.6f}")

print("\nSummary")
print(f"Non-zero orig->aug1 in {non_zero_orig_aug}/{num_trials} trials")
print(f"Non-zero aug1->aug2 in {non_zero_aug_aug}/{num_trials} trials")

if non_zero_orig_aug == 0 and non_zero_aug_aug == 0:
    print("Augmentation appears inactive for this sample.")
else:
    print("Augmentation is active.")

Trial 01: orig->aug1=0.000000, aug1->aug2=0.000000
Trial 02: orig->aug1=0.335406, aug1->aug2=0.000000
Trial 03: orig->aug1=0.335406, aug1->aug2=0.000000
Trial 04: orig->aug1=0.335406, aug1->aug2=0.000000
Trial 05: orig->aug1=0.000000, aug1->aug2=0.000000
Trial 06: orig->aug1=0.000000, aug1->aug2=0.000000
Trial 07: orig->aug1=0.335406, aug1->aug2=0.000000
Trial 08: orig->aug1=0.335406, aug1->aug2=0.335406
Trial 09: orig->aug1=0.000000, aug1->aug2=0.000000
Trial 10: orig->aug1=0.000000, aug1->aug2=0.000000

Summary
Non-zero orig->aug1 in 5/10 trials
Non-zero aug1->aug2 in 1/10 trials
Augmentation is active.


In [8]:
# -----------------------------------------------------------------------------
# Train
# -----------------------------------------------------------------------------

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)           # Ensure output directory exists
best_model_path = OUTPUT_DIR / "basic_cnn.pth"          # Path to save the best model based on validation loss

best_val_loss = float("inf")                            # Initialize best validation loss to infinity for early stopping
wait = 0                                                # Counter for early stopping patience

for epoch in range(EPOCHS):                             # Loop over the number of epochs to train the model
    net.train()                                         # Set the model to training mode, which enables features like dropout and batch normalization to behave appropriately during training
    running_loss = 0.0                                  # Initialize running loss for the epoch     

    for batch_images, batch_labels in train_loader:     # Loop over batches of training data from the DataLoader
        batch_images = batch_images.to(device)          # Move the batch of images to the specified device (GPU or CPU) for computation
        batch_labels = batch_labels.to(device)          # Move the batch of labels to the specified device (GPU or CPU) for computation

        optimizer.zero_grad()                           # Zero the gradients of the model parameters to prevent accumulation from previous batches                    

        outputs = net(batch_images)                     # Forward pass: compute the model's predictions for the current batch of images
        loss = loss_fn(outputs, batch_labels)           # Compute the loss between the model's predictions and the true labels using the specified loss function (CrossEntropyLoss in this case)
        loss.backward()                                 # Backward pass: compute the gradients of the loss with respect to the model parameters            
        optimizer.step()                                # Update the model parameters using the computed gradients and the optimization algorithm (Adam in this case)

        running_loss += loss.item()                     # Accumulate the loss for the current batch into the running loss for the epoch         

    train_loss = running_loss / len(train_loader)       # Compute the average training loss for the epoch by dividing the total running loss by the number of batches

    net.eval()                                          # Set the model to evaluation mode, which disables features like dropout and batch normalization to behave appropriately during evaluation  
    val_loss_total = 0.0                                # Initialize total validation loss for the epoch
    correct = 0                                        
    total = 0                                           

    with torch.no_grad():                               # Disable gradient computation during validation to save memory and computation since we are not updating the model parameters
        for batch_images, batch_labels in val_loader:   # Loop over batches of validation data from the DataLoader
            batch_images = batch_images.to(device)
            batch_labels = batch_labels.to(device)

            outputs = net(batch_images)
            loss = loss_fn(outputs, batch_labels)
            val_loss_total += loss.item()

            _, predicted = torch.max(outputs, 1)                # Get the predicted class indices by taking the index of the maximum logit for each sample in the batch
            total += batch_labels.size(0)
            correct += (predicted == batch_labels).sum().item() # Count the number of correct predictions by comparing the predicted class indices with the true labels and summing the number of matches

    val_loss = val_loss_total / len(val_loader)
    val_accuracy = correct / total

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        wait = 0
        torch.save(net.state_dict(), best_model_path)
    else:
        wait += 1
        if wait >= PATIENCE:
            print("Early stopping.")
            break

Epoch 01/30 | Train Loss: 0.8153 | Val Loss: 0.7498 | Val Acc: 0.6739
Epoch 02/30 | Train Loss: 0.6406 | Val Loss: 0.7044 | Val Acc: 0.6630
Epoch 03/30 | Train Loss: 0.6038 | Val Loss: 0.7355 | Val Acc: 0.6848
Epoch 04/30 | Train Loss: 0.5714 | Val Loss: 0.6515 | Val Acc: 0.7283
Epoch 05/30 | Train Loss: 0.5079 | Val Loss: 0.6281 | Val Acc: 0.6848
Epoch 06/30 | Train Loss: 0.4962 | Val Loss: 0.5980 | Val Acc: 0.7283
Epoch 07/30 | Train Loss: 0.4508 | Val Loss: 0.8435 | Val Acc: 0.6522
Epoch 08/30 | Train Loss: 0.4801 | Val Loss: 0.6959 | Val Acc: 0.7283
Epoch 09/30 | Train Loss: 0.4404 | Val Loss: 0.7221 | Val Acc: 0.7174
Epoch 10/30 | Train Loss: 0.4336 | Val Loss: 0.6157 | Val Acc: 0.7174
Epoch 11/30 | Train Loss: 0.4122 | Val Loss: 0.6768 | Val Acc: 0.7283
Epoch 12/30 | Train Loss: 0.4460 | Val Loss: 0.8001 | Val Acc: 0.6522
Epoch 13/30 | Train Loss: 0.4078 | Val Loss: 0.6635 | Val Acc: 0.7065
Epoch 14/30 | Train Loss: 0.3975 | Val Loss: 0.5828 | Val Acc: 0.7174
Epoch 15/30 | Train 

In [9]:
# -----------------------------------------------------------------------------
# Test
# -----------------------------------------------------------------------------

net = VegetationCNN().to(device)                    # Initialize the model and move it to the specified device
net.load_state_dict(torch.load(best_model_path, map_location=device, weights_only=True))
net.eval()

all_true = []
all_pred = []

with torch.no_grad():                              # Disable gradient computation during testing to save memory and computation since we are not updating the model parameters
    for batch_images, batch_labels in test_loader:  
        batch_images = batch_images.to(device)

        outputs = net(batch_images)
        _, predicted = torch.max(outputs, 1)       # Get the predicted class indices by taking the index of the maximum logit for each sample in the batch

        all_true.extend(batch_labels.numpy())      # Extend the all_true list with the true labels from the current batch by converting the batch_labels tensor to a numpy array and extending the list
        all_pred.extend(predicted.cpu().numpy())   # Move predicted labels to CPU and convert to numpy array before extending the all_pred list

accuracy = accuracy_score(all_true, all_pred)
precision = precision_score(all_true, all_pred, average="macro", zero_division=0)
recall = recall_score(all_true, all_pred, average="macro", zero_division=0)
f1 = f1_score(all_true, all_pred, average="macro", zero_division=0)

print("\nTest Results")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-Score : {f1:.4f}")

with open(OUTPUT_DIR / "basic_results.txt", "w") as file:
    file.write(f"Accuracy : {accuracy:.4f}\n")
    file.write(f"Precision: {precision:.4f}\n")
    file.write(f"Recall   : {recall:.4f}\n")
    file.write(f"F1-Score : {f1:.4f}\n")

print(f"\nSaved model: {best_model_path}")




Test Results
Accuracy : 0.8495
Precision: 0.9034
Recall   : 0.6625
F1-Score : 0.7066

Saved model: c:\Users\Public\OpenFrameworks\apps\myApps\PandaHat\Learning_Path\Task_4\outputs\basic_cnn.pth


## Test Results Summary

### Best Result
- **Horizontal flip (0.5) + Batch Normalization + extended patience with max epochs (30)**
  - Accuracy : 0.8495
  - Precision: 0.9034
  - Recall   : 0.6625
  - F1-Score : 0.7066

### Task 4 Results Ordered (Best to Worst)
**Legend:** 
- HF = Horizontal Flip
- BN = Batch Normalization 
- P = Patience
- R = ReLU  
- S = Sigmoid
- E = Epochs
- K = Kernerl Size

| Configuration | Accuracy | Precision | Recall | F1-Score |
|---|---|---|---|---|
| R + HF (0.5) + BN + P + E (30) + K (3x3) | 0.8495 | 0.9034 | 0.6625 | 0.7066 |
| R + HF (0.5) + BN + P + E (15) + K (3x3) | 0.8172 | 0.8816 | 0.6563 | 0.6860 |
| R + HF (1.0) + BN + P + E (15) + K (3x3) | 0.7957 | 0.8643 | 0.6387 | 0.6713 |
| R + HF (0.5) + BN + P + E (30) + K (5x5) | 0.8065 | 0.5409 | 0.5573 | 0.5480 |
| S + HF (0.5) + BN + P + E (30) + K (3x3) | 0.7849 | 0.5326 | 0.5375 | 0.5310 |
| R + HF (1.0) + E (15) + K (3x3) | 0.6989 | 0.4676 | 0.4919 | 0.4756 |
| R + HF + Rotation + E (15) + K (3x3) | 0.6989 | 0.4648 | 0.4897 | 0.4754 |
| R + No Augmentation (Task 3 Comparison) | 0.6882 | 0.4557 | 0.4742 | 0.4646 |
| R + HF + Rotation + Color Jitter + E (15) + K (3x3)  | 0.6344 | 0.4300 | 0.4214 | 0.4109 |
| R + Heavy Augmentation (All) + E (15) + K (3x3)| 0.5484 | 0.1828 | 0.3333 | 0.2361 |

### Task 3 Results (No Augmentation Data)

| Configuration | Accuracy | Precision | Recall | F1-Score |
|---|---|---|---|---|
| No Augmentation | 0.6882 | 0.4557 | 0.4742 | 0.4646 |

### Task 2 Results (SVM and CNN Data)

| Configuration | Accuracy | Precision | Recall | F1-Score |
|---|---|---|---|---|
| SVM | 0.8745 | 0.6400 | 0.5800 | 0.6000 |
| CNN | 0.6855 | N/A | N/A | N/A |

### Key Observations
- **Too much augmentation hurts performance** - The combination of rotation, vertical flip, color jitter, and random crop reduced accuracy significantly
- **Moderate horizontal flip (0.5) is optimal** - Better than both no augmentation and aggressive flipping (1.0)
- **Batch normalization + patience are essential** - These significantly improved results
- **More epochs helped** - The model improved up to the current 30-epoch cap before early stopping kicked in
- **Imbalanced data** - The training data is imbalanced

### Needed Improvements
- **Weighted samplers** - So minority classes appear more often in training batches
- **Better tuning for recall** - Search for ways to tune the CNN or data to get better recall